## __PHÂN TÍCH HIỆU SUẤT GIAO HÀNG LOGISTIC__

### __1. Mô Tả Dataset__

Bộ dữ liệu ghi lại nhật ký **theo dõi (tracking) các chuyến vận tải đường bộ** của một mạng lưới
logistics tại **Ấn Độ**, chủ yếu phục vụ các nhà máy ô tô/công nghiệp (Daimler, Ford, Ashok Leyland,
Larsen & Toubro…). Mỗi dòng là **một chuyến hàng** với đầy đủ hành trình từ lúc đặt (booking) đến lúc
giao, kèm dữ liệu GPS.

| Thuộc tính | Giá trị |
|---|---|
| Grain (1 dòng = ?) | **1 chuyến hàng**, khóa  Booking ID  |
| Số dòng | **3.585**  |
| Số cột | **28** |
| Khoảng thời gian | **15/04/2019 -> 03/12/2020**, nhưng khối lượng dồn vào **06-08/2020** |
| Bối cảnh | Logistics nội địa Ấn Độ (địa danh & tuyến trong nước) |

---

Dữ liệu có thể chia thành **4 nhóm**:

1. **Thông tin vận chuyển** — mã đơn, loại hình, thời điểm booking, các mốc thời gian kế hoạch/thực tế (ETA, trip start/end).
2. **GPS & vận hành** — nhà cung cấp GPS, tọa độ điểm đi/đến/hiện tại, thời điểm ping, quãng đường.
3. **Đối tác & khách hàng** — nhà xe (supplier), khách nhận hàng (customer), tài xế, biển số xe, loại xe.
4. **Vật liệu** — loại hàng hóa được vận chuyển.

Danh sách các cột


| Cột | Mô tả |
|---|---|
| GpsProvider | Nhà cung cấp dịch vụ GPS theo dõi phương tiện. |
| BookingID | Mã định danh duy nhất cho mỗi lần đặt chuyến (booking). |
|  Shipment Type  | Cho biết chuyến là **Market** (đặt lẻ / spot) hay **Regular** (theo hợp đồng). |
|  BookingID_Date  | Ngày và giờ tạo booking. |
|  Vehicle Registration  | Số đăng ký (biển số) duy nhất của xe dùng để vận chuyển. |
|  Origin_Location  | Điểm xuất phát ban đầu của chuyến hàng. |
|  Destination_Location  | Điểm đến cuối cùng nơi hàng được giao. |
|  Origin_loc_latitude  | Vĩ độ của điểm xuất phát. |
|  Origin_loc_longitude  | Kinh độ của điểm xuất phát. |
|  Destination_loc_latitude  | Vĩ độ của điểm đến. |
|  Destination_loc_longitude  | Kinh độ của điểm đến. |
|  Data_Ping_time  | Thời điểm bản ghi GPS (ping) gần nhất từ xe. |
|  Planned_ETA  | Thời gian đến dự kiến tại điểm đến theo kế hoạch chuyến đi. |
|  Current_Location  | Vị trí gần nhất được ghi nhận của xe. |
|  actual_eta  | Thời gian đến thực tế tại điểm đến. |
|  Current_loc_latitude  | Vĩ độ vị trí hiện tại của xe. |
|  Current_loc_longitude  | Kinh độ vị trí hiện tại của xe. |
|  ontime  | Cho biết xe có đến đúng giờ hay không (Yes/No). |
|  trip_start_date  | Ngày và giờ bắt đầu chuyến đi thực tế. |
|  trip_end_date  | Ngày và giờ đến (ước tính). |
|  TRANSPORTATION_DISTANCE_IN_KM  | Tổng quãng đường xe đã đi trong chuyến, tính bằng km. |
|  vehicleType  | Loại xe dùng cho chuyến hàng (ví dụ: 32 FT Truck, Tata Ace, Multi-Axle). |
|  Minimum_kms_to_be_covered_in_a_day  | Quãng đường tối thiểu xe được kỳ vọng đi trong một ngày. |
|  Driver_Name  | Tên tài xế được phân công cho chuyến. |
|  Driver_MobileNo  | Số điện thoại liên hệ của tài xế. |
|  customerNameCode  | Tên khách hàng nhận hàng. |
|  supplierNameCode  | Tên nhà cung cấp / đơn vị gửi hàng (nhà xe). |
|  Material Shipped  | Mô tả loại hàng hóa được vận chuyển trong chuyến. |



### __2. Import thư viện__

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path

### __3. Tổng quan dữ liệu__

In [23]:
DATA_PATH = Path('../data/raw/Transportation & Logistics Tracking Dataset.xlsx')
df_raw = pd.read_excel(DATA_PATH,sheet_name="Primary Data")
df_raw.head()

,Gps Provider,Booking ID,Shipment Type,Booking Date,Vehicle Registration,Origin Location,Destination Location,Origin Location Latitude,Origin Location Longitude,Destination Location Latitude,...,Trip Start Date,Trip End Date,Transportation Distance (KM),Vehicle Type,Minimum Kms To Be Covered In A Day,Driver Name,Driver Mobile No,Customer Name,Supplier Name,Material Shipped
0,Consent Track,AEIBK2027469,Regular,2020-08-26 12:03:46.000,MH14GD9464,"Shive, Pune, Maharashtra","Pondur, Kanchipuram, Tamil Nadu",18.750621,73.877190,12.930429,...,2020-08-26 16:16:00,2020-08-28 12:15:10,1290.0,32 FT Multi-Axle 14MT - HCV,NaN,VIRAT NILAPALLE,9.960008e+09,Daimler India Commercial Vehicles Pvt Lt,Oms Logistics Pvt Ltd,Regulator - 12v
1,Vamosys,VCV00014153/082021,Regular,2020-08-27 15:21:48.570,TN30BC9320,"Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,...",12.839000,79.954000,12.839000,...,2020-08-27 15:21:06,2020-08-27 15:21:54.947000,29.0,NaN,NaN,SENTHIL KUMAR,NaN,Daimler India Commercial Vehicles Pvt Lt,Vj Logistics,Valve Spring
2,Vamosys,VCV00014063/082021,Regular,2020-08-27 14:22:17.833,TN30BB1036,"Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,...",12.839000,79.954000,12.839000,...,2020-08-27 14:21:21,2020-08-27 14:22:25.037000,21.0,NaN,NaN,ANBU,NaN,Daimler India Commercial Vehicles Pvt Lt,Vj Logistics,Valve Spring
3,Vamosys,VCV00014741/082021,Regular,2020-08-28 00:32:20.523,TN88D4134,"Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,...",12.839000,79.954000,12.839000,...,2020-08-28 00:31:41,2020-08-28 00:32:24.213000,20.0,NaN,NaN,SUDHAKAR,NaN,Daimler India Commercial Vehicles Pvt Lt,Namakkal Sri Anjinaya Transport,Lu Hood Lock / Rh
4,Consent Track,AEIBK2027446,Regular,2020-08-26 09:55:08.000,GJ01DZ8943,"Khorajnanoda, Ahmedabad, Gujarat","Singaperumalkoil, Kanchipuram, Tamil Nadu",22.961777,72.094219,12.786517,...,2020-08-26 15:35:00,2020-08-28 11:21:00,1900.0,32 FT Single-Axle 7MT - HCV,NaN,MAN SINGH,6.396821e+09,Ford India Private Limited,Sterling Translogistics Private Limited,Lu Latch / Pin


In [24]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3585 entries, 0 to 3584
Data columns (total 28 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   Gps Provider                        3585 non-null   object        
 1   Booking ID                          3585 non-null   object        
 2   Shipment Type                       3585 non-null   object        
 3   Booking Date                        3585 non-null   datetime64[ns]
 4   Vehicle Registration                3585 non-null   object        
 5   Origin Location                     3585 non-null   object        
 6   Destination Location                3585 non-null   object        
 7   Origin Location Latitude            3585 non-null   float64       
 8   Origin Location Longitude           3585 non-null   float64       
 9   Destination Location Latitude       3585 non-null   float64       
 10  Destination Location Lon

In [25]:
df_raw.shape

(3585, 28)

In [26]:
df_raw.describe().T

,count,mean,min,25%,50%,75%,max,std
Booking Date,3585,2020-06-21 21:10:33.783650560,2019-04-15 15:15:13,2020-06-12 16:15:00,2020-07-23 13:50:18,2020-08-14 23:34:17.889999872,2020-12-03 13:10:21,NaN
Origin Location Latitude,3585.0,17.870904,9.973636,12.839,16.560192,22.961777,30.000345,5.688552
Origin Location Longitude,3585.0,78.8575,72.056,76.835337,79.632,80.184717,91.843582,4.48121
Destination Location Latitude,3585.0,19.049239,8.172701,12.839,18.660455,23.953847,32.684722,6.390109
Destination Location Longitude,3585.0,78.928819,70.740636,76.835337,78.099421,79.975221,94.961065,4.450874
Data Ping time,3584,2020-06-29 19:31:07.833147392,2019-06-14 15:20:12,2020-06-25 14:52:38.750000128,2020-07-29 17:35:13,2020-08-18 00:05:08,2020-08-28 12:15:10,NaN
Current Location Latitude,3584.0,18.505396,8.70089,12.835129,16.964161,23.283158,32.367928,6.130438
Current Location Longitude,3584.0,78.947352,69.657698,76.854271,78.20903,80.019062,95.52955,4.291132
Transportation Distance (KM),3437.0,841.10032,0.0,107.0,400.0,1290.0,2898.0,851.889681
Minimum Kms To Be Covered In A Day,940.0,250.531915,250.0,250.0,250.0,250.0,275.0,3.609543


> **Nhận xét ban đầu:**
>
> - Dataset thô có **3.585 dòng và 28 cột**.
> - `Booking Date` và `Data Ping time` đã được Pandas nhận diện dưới dạng `datetime64[ns]`.
> - Một số cột thời gian quan trọng như `Planned ETA`, `Actual ETA`, `Trip Start Date` và `Trip End Date` đang có kiểu `object`, cho thấy dữ liệu thời gian chưa hoàn toàn đồng nhất và cần được kiểm tra kỹ hơn.
> - 3.585 dòng dữ liệu chưa được xem là 3.585 booking. 

### __4. Kiểm tra dữ liệu__

In [27]:
df = df_raw.copy()

#### __4.1 Kiểm tra grain và duplicated theo 'Booking ID'__

In [28]:
# Kiểm tra tính duy nhất của Booking ID
print(f'Số dòng dữ liệu thô: {len(df_raw):,}')
print(f"Số Booking ID duy nhất: {df_raw['Booking ID'].nunique():,}")
print(f'Số dòng duplicated hoàn toàn: {df_raw.duplicated().sum():,}')

Số dòng dữ liệu thô: 3,585
Số Booking ID duy nhất: 3,582
Số dòng duplicated hoàn toàn: 0


>Dataset có **3585 dòng dữ liệu thô** nhưng chỉ có **3582 Booking ID duy nhất**  
>  Không có dòng nào bị duplicate hoàn toàn trên tất cả các cột.

In [29]:
# Drill down - tìm Booking ID xuất hiện nhiều lần
booking_counts = (df_raw['Booking ID'].value_counts())
duplicate_booking_ids = booking_counts[booking_counts > 1]
duplicate_booking_ids

Booking ID
MVCV0000798/082021    3
MVCV0000759/082021    2
Name: count, dtype: int64

In [30]:
extra_booking_rows = (duplicate_booking_ids - 1).sum()
print(f"Số Booking ID xuất hiện nhiều lần: {len(duplicate_booking_ids)}")
print(f"Số dòng vượt quá grain 1 row/booking: {extra_booking_rows}")

Số Booking ID xuất hiện nhiều lần: 2
Số dòng vượt quá grain 1 row/booking: 3


In [31]:
# Kiểm tra xem các dòng dữ liệu đó khác nhau như thế nào
duplicate_booking_rows = df_raw[df_raw['Booking ID'].isin(
                            duplicate_booking_ids.index)].sort_values(
                                ['Booking ID', 'Customer Name', 'Material Shipped'] 
                            )
duplicate_booking_rows[[
        "Booking ID",
        "Customer Name",
        "Supplier Name",
        "Material Shipped",
        "Planned ETA",
        "Actual ETA",
        "Origin Location",
        "Destination Location"
    ]]

,Booking ID,Customer Name,Supplier Name,Material Shipped,Planned ETA,Actual ETA,Origin Location,Destination Location
820,MVCV0000759/082021,Praveen Engineering Products India Pvt L,A.V.Transports,Rectifier,2020-08-17 19:58:26,2020-08-19 00:51:51.150000,"Praveen Engineering Industries,Hosur,Tamil Nadu","Sri Devi Powder Coating Industries,Chennai,Tam..."
821,MVCV0000759/082021,Praveen Engineering Products India Pvt L,A.V.Transports,Rectifier Arrangement,2020-08-17 19:58:26,2020-08-19 00:51:51.150000,"Praveen Engineering Industries,Hosur,Tamil Nadu","Sri Devi Powder Coating Industries,Chennai,Tam..."
881,MVCV0000798/082021,Hi-Tech Gears Ltd,Dhillon Goods Transport,Iut Part,2020-08-18 14:52:06,2020-08-18 15:50:10.663000,"Tvslsl-Jamalpurl-Hub,Gurgaon,Haryana","Hi-Tech Gears Ltd,Satara,Maharashtra"
882,MVCV0000798/082021,Rico Auto Industries Ltd,Dhillon Goods Transport,Sol. Relay With Cable Assy,2020-08-18 14:52:06,2020-08-18 15:50:10.910000,"Tvslsl-Jamalpurl-Hub,Gurgaon,Haryana","Hi-Tech Gears Ltd,Satara,Maharashtra"
883,MVCV0000798/082021,Talbros Automotive Components Ltd,Dhillon Goods Transport,Solenoid Switch,2020-08-18 14:52:06,2020-08-18 15:50:10.663000,"Tvslsl-Jamalpurl-Hub,Gurgaon,Haryana","Hi-Tech Gears Ltd,Satara,Maharashtra"


>- Có **2 Booking ID xuất hiện nhiều hơn một lần**, tạo ra tổng cộng **3 dòng vượt quá grain 1 dòng/booking**.
>- Các dòng có cùng Booking ID không hoàn toàn giống nhau cũng có thể đơn đó chia ra làm hai chuyến.
>- Vì vậy, không nên xóa duplicate theo `Booking ID` ngay lập tức vì có nguy cơ làm mất thông tin.

#### __4.2 Kiểm tra dữ liệu thiếu__

In [32]:
missing_audit = (pd.DataFrame({
    'missing_count': df_raw.isna().sum(),
    'missing_pct': df_raw.isna().mean() * 100
})).query('missing_count > 0').sort_values('missing_pct', ascending=False)

missing_audit['missing_pct'] = (
    missing_audit['missing_pct'].round(2)
)

missing_audit

,missing_count,missing_pct
Minimum Kms To Be Covered In A Day,2645,73.78
Driver Mobile No,1024,28.56
Vehicle Type,764,21.31
Driver Name,317,8.84
Transportation Distance (KM),148,4.13
Actual ETA,25,0.70
Data Ping time,1,0.03
Current Location Longitude,1,0.03
Current Location Latitude,1,0.03


In [33]:
# Có những dữ liệu thì k bị thiếu nhưng thực tế trong dữ liệu có NULL
NULL_LIKE_VALUES = {
    'NULL',
    'NA',
    'N/A',
    'NONE',
    '-'
}

null_like_audit = {}
for col in df_raw.select_dtypes(include='object').columns:
    cleaned_values = (
        df_raw[col].astype('string').str.strip().str.upper()
    )

    count = cleaned_values.isin(NULL_LIKE_VALUES).sum()
    if count > 0:
        null_like_audit[col] = count

pd.Series(
    null_like_audit,
    name= 'null_like_count'
).sort_values(ascending=False)

Current Location    12
Gps Provider         1
Name: null_like_count, dtype: int64

>Kết quả audit cho thấy dữ liệu thiếu không chỉ tồn tại dưới dạng `NaN`, mà còn xuất hiện dưới dạng chuỗi `"NULL"` trong một số cột text. Do đó, chỉ sử dụng `df.isna()` sẽ chưa phản ánh đầy đủ tình trạng missing data. Trong bước Data Cleaning, các giá trị dạng `"NULL"`, `"NA"`, `"N/A"`, `"NONE"` hoặc `"-"` sẽ được chuẩn hóa về missing value trước khi xử lý tiếp.

#### __4.3 Phân loại missing values theo mức độ ảnh hưởng của bussiness__
>Missing này thì ảnh hưởng gì đến business question và các KPIs?

- Ta thấy rằng cột Actual ETA cực kì quan trong vì KPI của project dựa trên: **Actual ETA - Planned ETA**.

In [34]:
# Kiểm tra missing values cho Actual ETA
missing_actual_eta = df_raw[df_raw['Actual ETA'].isna()]
print(f'Số dòng thiếu Actual ETA: {len(missing_actual_eta):,}')

missing_actual_eta[[
        "Booking ID",
        "Planned ETA",
        "Actual ETA",
        "Ontime",
        "Trip Start Date",
        "Trip End Date"
    ]].head(10)


Số dòng thiếu Actual ETA: 25


,Booking ID,Planned ETA,Actual ETA,Ontime,Trip Start Date,Trip End Date
1320,AEIBK2025104,2020-08-08 03:32:00,NaN,No,2020-08-07 11:27:20,2020-08-11 14:35:00
1328,AEIBK2025225,2020-08-12 16:06:24,NaN,Yes,2020-08-08 12:06:24,2020-08-11 13:51:00
1372,AEIBK2024733,2020-08-08 17:16:33,NaN,No,2020-08-04 13:16:33,2020-08-10 23:18:00
1378,AEIBK2024804,2020-08-08 20:19:51,NaN,No,2020-08-04 16:19:51,2020-08-10 17:43:00
1381,AEIBK2025095,2020-08-11 14:49:33,NaN,Yes,2020-08-07 10:49:33,2020-08-10 16:49:00
1393,AEIBK2024805,2020-08-08 20:31:33,NaN,No,2020-08-04 16:31:33,2020-08-10 12:36:00
1398,AEIBK2025085,2020-08-11 13:53:49,NaN,Yes,2020-08-07 09:53:49,2020-08-10 11:04:00
1400,AEIBK2024667,2020-08-07 16:15:13,NaN,No,2020-08-03 12:15:13,2020-08-10 09:53:00
1401,AEIBK2025125,2020-08-07 16:46:08,NaN,No,2020-08-07 15:56:38,2020-08-10 09:26:00
1658,AEIBK2024300,2020-08-03 19:43:32,NaN,No,2020-07-30 15:43:32,2020-08-04 09:20:00


>Dựa vào kết quả, ta nhận thấy rằng nếu không có **Actual ETA** thì cột **Ontime** vẫn có kết quả. Điều này cho thấy rằng không thể independently validate Ontime bằng ETA ở 25 booking này.


- Cột `Transportation Distance (KM)` thiếu khoảng **4.13%**, bị thiếu ở một số booking; các booking này vẫn có thể sử dụng cho KPI tổng thể nhưng không phù hợp cho distance analysis.
- `Vehicle Type` thiếu khoảng **21.31** có tỷ lệ missing đáng kể nhưng không nên loại bỏ toàn bộ các booking này vì chúng vẫn có giá trị cho các KPI khác.

- Bảng đánh giá tác động của missing values

| Cột | Mức độ thiếu | Ảnh hưởng | Hướng xử lý dự kiến |
|---|---:|---|---|
| Minimum Kms To Be Covered In A Day | Cao | Không phải biến trọng tâm của project | Xem xét loại khỏi analytical dataset |
| Driver Mobile No | Cao | Không phục vụ business analysis | Loại khỏi analytical dataset |
| Vehicle Type | Cao | Ảnh hưởng Vehicle Analysis | Giữ booking và gán nhóm `Unknown` sau cleaning |
| Driver Name | Trung bình | Không phải KPI chính | Đánh giá nhu cầu sử dụng trước khi xử lý |
| Transportation Distance (KM) | Thấp | Ảnh hưởng Distance Analysis | Giữ booking, loại khỏi các phân tích cần distance nếu thiếu |
| Actual ETA | Thấp nhưng quan trọng | Không tính được Delay KPI | Không đưa vào Eligible Bookings |
| Current Location Latitude / Longitude | Rất thấp | Chủ yếu ảnh hưởng location analysis | Đánh giá sau |
| Data Ping time | Rất thấp | Không ảnh hưởng trực tiếp KPI delay | Đánh giá sau |
| `"NULL"` dạng text | Có tồn tại | `isna()` không phát hiện | Chuẩn hóa về `NaN` trong cleaning |

#### __4.4 Kiểm tra chất lượng dữ liệu theo thời gian__

In [35]:
DATETIME_COLS_RAW = [
    "Booking Date",
    "Data Ping time",
    "Planned ETA",
    "Actual ETA",
    "Trip Start Date",
    "Trip End Date"]

df_raw[DATETIME_COLS_RAW].dtypes

Booking Date       datetime64[ns]
Data Ping time     datetime64[ns]
Planned ETA                object
Actual ETA                 object
Trip Start Date            object
Trip End Date              object
dtype: object

> Ta nhận thấy thấy rằng có 4 kiểu dữ liệu chưa đồng nhất vì thế nên cần kiểm tra sâu hơn

In [36]:
datetime_object_cols = [
    "Planned ETA",
    "Actual ETA",
    "Trip Start Date",
    "Trip End Date"]

for col in datetime_object_cols:
    print(f"\n-{col}-")
    print(df_raw[col].map(lambda x: type(x).__name__).value_counts(dropna=False))


-Planned ETA-
Planned ETA
datetime    3583
str            2
Name: count, dtype: int64

-Actual ETA-
Actual ETA
datetime    3558
float         25
str            2
Name: count, dtype: int64

-Trip Start Date-
Trip Start Date
datetime    3583
str            2
Name: count, dtype: int64

-Trip End Date-
Trip End Date
datetime    3583
str            2
Name: count, dtype: int64


> Có 2 record dạng string làm cho cả cột thành object

In [37]:
# Kiểm tra khả năng parse datetime
datetime_audit = []

for col in DATETIME_COLS_RAW:
    parsed = pd.to_datetime(df_raw[col],errors="coerce")

    original_missing = df_raw[col].isna().sum()
    parsed_missing = parsed.isna().sum()

    datetime_audit.append({
        "column": col,
        "original_dtype": str(df_raw[col].dtype),
        "missing_before": original_missing,
        "missing_after_parse": parsed_missing,
        "parse_failed": parsed_missing - original_missing,
        "min_datetime": parsed.min(),
        "max_datetime": parsed.max(),
        "before_2000": (
            (parsed.dt.year < 2000) & parsed.notna()).sum()
        })

datetime_audit = pd.DataFrame(datetime_audit)
datetime_audit

,column,original_dtype,missing_before,missing_after_parse,parse_failed,min_datetime,max_datetime,before_2000
0,Booking Date,datetime64[ns],0,0,0,2019-04-15 15:15:13,2020-12-03 13:10:21.000,0
1,Data Ping time,datetime64[ns],1,1,0,2019-06-14 15:20:12,2020-08-28 12:15:10.000,0
2,Planned ETA,object,0,0,0,1899-12-30 04:06:00,2020-12-05 00:57:28.000,2
3,Actual ETA,object,25,25,0,1899-12-30 03:21:00,2020-08-29 08:37:27.420,2
4,Trip Start Date,object,0,0,0,1899-12-30 00:00:00,2020-12-03 13:10:21.000,2
5,Trip End Date,object,0,0,0,1899-12-30 03:21:00,2020-08-28 12:15:10.000,2


In [38]:
# Kiểm tra xem Booking ID nào là bất thường
parsed_datetime = {}

for col in DATETIME_COLS_RAW:
    parsed_datetime[col] = pd.to_datetime(df_raw[col], errors="coerce")

historical_mask = pd.Series(False, index=df_raw.index)

for col in [
    "Planned ETA",
    "Actual ETA",
    "Trip Start Date",
    "Trip End Date"]:
    historical_mask |= (parsed_datetime[col].dt.year < 2000)

df_raw.loc[historical_mask,[
        "Booking ID",
        "Booking Date",
        "Planned ETA",
        "Actual ETA",
        "Trip Start Date",
        "Trip End Date",
        "Ontime"
    ]]

,Booking ID,Booking Date,Planned ETA,Actual ETA,Trip Start Date,Trip End Date,Ontime
3583,WDSBKTP49392,2019-06-10 13:17:44,1899-12-30 08:58:00.000,1899-12-30 08:13:00.000,1899-12-30 00:00:00.000,1899-12-30 08:13:00.000,Yes
3584,WDSBKTP44502,2019-04-15 15:15:13,1899-12-30 04:06:00.000,1899-12-30 03:21:00.000,1899-12-30 00:00:00.000,1899-12-30 03:21:00.000,No


>- Các giá trị hiện có có thể được parse bằng `pd.to_datetime()` mà không phát sinh thêm parse failure. Tuy nhiên, khả năng parse thành công không đảm bảo giá trị thời gian hợp lệ về mặt nghiệp vụ.  
>- Và ta thấy rằng, có 2 booking chứa timestamp bất thường năm 1899 trong các trường thời gian quan trọng. Cũng có thể là khi dữ liệu bị null hệ thống tự mặc định là thời gian đó.

In [39]:
# Kiểm tra logic về thời gian
booking_dt = pd.to_datetime(df_raw["Booking Date"], errors="coerce")
planned_dt = pd.to_datetime(df_raw["Planned ETA"], errors="coerce")
actual_dt = pd.to_datetime(df_raw["Actual ETA"], errors="coerce")
trip_start_dt = pd.to_datetime(df_raw["Trip Start Date"], errors="coerce")
trip_end_dt = pd.to_datetime(df_raw["Trip End Date"], errors="coerce")

# Tạm loại timestamp lịch sử ra khỏi 
for s in [
    planned_dt,
    actual_dt,
    trip_start_dt,
    trip_end_dt
]:
    s.loc[s.dt.year < 2000] = pd.NaT

datetime_rule_audit = pd.Series({
    "actual_before_booking":
        (actual_dt < booking_dt).sum(),
    "actual_before_trip_start":
        (actual_dt < trip_start_dt).sum(),
    "planned_before_trip_start":
        (planned_dt < trip_start_dt).sum(),
    "trip_end_before_trip_start":
        (trip_end_dt < trip_start_dt).sum()
})

datetime_rule_audit

actual_before_booking          7
actual_before_trip_start       7
planned_before_trip_start     43
trip_end_before_trip_start    57
dtype: int64

>Có 7 record có `Actual ETA` xảy ra trước `Booking Date` và `Trip Start Date`, đây là các bất thường về trình tự thời gian cần được flag trước khi tính KPI.

In [40]:
invalid_actual_mask = (
    (actual_dt < booking_dt) |
    (actual_dt < trip_start_dt)
)

df_raw.loc[invalid_actual_mask,
    [
        "Booking ID",
        "Booking Date",
        "Planned ETA",
        "Actual ETA",
        "Trip Start Date",
        "Trip End Date",
        "Ontime"
    ]]

,Booking ID,Booking Date,Planned ETA,Actual ETA,Trip Start Date,Trip End Date,Ontime
1666,AEIBK2017875,2020-12-03 13:10:21,2020-12-05 00:57:28,2020-08-05 07:43:55.383000,2020-12-03 13:10:21,2020-03-12 16:23:00,Yes
1850,AEIBK2017783,2020-10-03 19:05:53,2020-10-07 23:05:53,2020-07-29 18:19:54.237000,2020-10-03 19:05:53,2020-03-14 11:43:00,Yes
2391,AEIBK2017672,2020-08-03 02:56:05,2020-08-07 06:56:05,2020-07-16 03:16:53.167000,2020-08-03 02:56:05,2020-03-12 18:04:00,Yes
2392,AEIBK2017729,2020-09-03 16:31:57,2020-09-04 14:05:39,2020-07-14 14:29:03.550000,2020-09-03 16:31:57,2020-03-12 20:31:00,Yes
2393,AEIBK2017781,2020-10-03 18:26:21,2020-10-07 22:26:21,2020-07-14 14:17:20.017000,2020-10-03 18:26:21,2020-03-14 11:43:00,Yes
2394,AEIBK2017785,2020-10-03 20:05:28,2020-10-08 00:05:28,2020-07-15 08:41:19.500000,2020-10-03 20:05:28,2020-03-14 16:30:00,Yes
2395,AEIBK2017791,2020-11-03 10:12:17,2020-11-07 14:12:17,2020-07-15 05:48:12.193000,2020-11-03 10:12:17,2020-03-13 23:00:00,Yes


>Ta thấy rằng có các đơn còn chưa được booking nhưng actual ETA lại xảy ra trước đó vài tháng

#### __4.5 Kiểm tra dữ liệu số__

In [42]:
# Các biến số thực sự dùng cho phân tích
NUMERIC_ANALYSIS_COLS = [
    "Transportation Distance (KM)",
    "Origin Location Latitude",
    "Origin Location Longitude",
    "Destination Location Latitude",
    "Destination Location Longitude",
    "Current Location Latitude",
    "Current Location Longitude",
    "Minimum Kms To Be Covered In A Day"
]

df_raw[NUMERIC_ANALYSIS_COLS].describe().T

,count,mean,std,min,25%,50%,75%,max
Transportation Distance (KM),3437.0,841.100320,851.889681,0.000000,107.000000,400.000000,1290.000000,2898.000000
Origin Location Latitude,3585.0,17.870904,5.688552,9.973636,12.839000,16.560192,22.961777,30.000345
Origin Location Longitude,3585.0,78.857500,4.481210,72.056000,76.835337,79.632000,80.184717,91.843582
Destination Location Latitude,3585.0,19.049239,6.390109,8.172701,12.839000,18.660455,23.953847,32.684722
Destination Location Longitude,3585.0,78.928819,4.450874,70.740636,76.835337,78.099421,79.975221,94.961065
Current Location Latitude,3584.0,18.505396,6.130438,8.700890,12.835129,16.964161,23.283158,32.367928
Current Location Longitude,3584.0,78.947352,4.291132,69.657698,76.854271,78.209030,80.019062,95.529550
Minimum Kms To Be Covered In A Day,940.0,250.531915,3.609543,250.000000,250.000000,250.000000,250.000000,275.000000


In [ ]:
# Kiểm tra Transportation Distance (KM) 
'''Đây là biến số quan trọng nhất vì nó liên quan 
đến việc phân tích KPI: "Khoảng cách có liên quan đến delay hay không?" '''
distance = df_raw["Transportation Distance (KM)"]

distance_audit = pd.Series({
    "count": distance.notna().sum(),
    "missing": distance.isna().sum(),
    "zero": (distance == 0).sum(),
    "negative": (distance < 0).sum(),
    "min": distance.min(),
    "median": distance.median(),
    "mean": distance.mean(),
    "max": distance.max()
})

distance_audit

count       3437.00000
missing      148.00000
zero          18.00000
negative       0.00000
min            0.00000
median       400.00000
mean         841.10032
max         2898.00000
dtype: float64

> Ta thấy rằng không có distance nào âm cả và có 18 `booking id` với distance = 0 nhưng chưa vội kết luận đó là lỗi.

In [ ]:
# Kiểm tra 18 booking có distance = 0
zero_distance_rows = df_raw.loc[
    df_raw["Transportation Distance (KM)"].eq(0),
    [
        "Booking ID",
        "Origin Location",
        "Destination Location",
        "Origin Location Latitude",
        "Origin Location Longitude",
        "Destination Location Latitude",
        "Destination Location Longitude",
        "Transportation Distance (KM)"
    ]]

zero_distance_rows.head()

,Booking ID,Origin Location,Destination Location,Origin Location Latitude,Origin Location Longitude,Destination Location Latitude,Destination Location Longitude,Transportation Distance (KM)
1117,AEIBK2024564,"Shive, Pune, Maharashtra","Shive, Pune, Maharashtra",18.750621,73.87719,18.750621,73.87719,0.0
1206,AEIBK2025418,"Shive, pune, maharashtra","Shive, Pune, Maharashtra",18.750621,73.87719,18.750621,73.87719,0.0
1207,AEIBK2024916,"Shive, pune, maharashtra","Shive, Pune, Maharashtra",18.750621,73.87719,18.750621,73.87719,0.0
1208,AEIBK2025420,"Shive, pune, maharashtra","Shive, Pune, Maharashtra",18.750621,73.87719,18.750621,73.87719,0.0
1214,AEIBK2025410,"Shive, pune, maharashtra","Shive, Pune, Maharashtra",18.750621,73.87719,18.750621,73.87719,0.0


> Sau khi drill down vào 18 `Booking ID` có distance = 0, ta thấy rằng các cột `Origin Location` và `Destination Location` bằng nhau. Điều này có khả năng đây là shipment nội bộ/cùng location. Do đó không drop 18 này chỉ vì distance = 0.

In [ ]:
# Kiểm tra Outlier của distance
distance.quantile(
    [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

0.01       3.0
0.05      20.0
0.10      26.0
0.25     107.0
0.50     400.0
0.75    1290.0
0.90    2400.0
0.95    2425.0
0.99    2700.0
Name: Transportation Distance (KM), dtype: float64

> Dãy số trên cho thấy dữ liệu khoảng cách có phân phối bị lệch phải nặng. 

In [ ]:
# Kiểm tra IQR outlier
q1 = distance.quantile(0.25)
q3 = distance.quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1: {q1}")
print(f"Q3: {q3}")
print(f"IQR: {iqr}")
print(f"Lower bound: {lower_bound}")
print(f"Upper bound: {upper_bound}")

print("High IQR outliers:",(distance > upper_bound).sum())

Q1: 107.0
Q3: 1290.0
IQR: 1183.0
Lower bound: -1667.5
Upper bound: 3064.5
High IQR outliers: 0


>Thấy rằng **High IQR outliers = 0** nghĩa là không có một dòng dữ liệu nào trong cột distance bị coi là giá trị lỗi (outlier) ở phía biên trên theo phương pháp IQR toán học.
> - Doanh nghiệp chạy cả chặng ngắn nội bang (107km) lẫn xuyên bang đường dài (1,290km), khiến khoảng biến động dữ liệu (IQR) rất rộng. Vì dải dữ liệu vốn đã rộng, công thức tự mở rộng vùng an toàn lên tới 3,064.5km.
> - Vì vậy, chuyến xe dài nhất 2,898km vẫn nằm trong vùng an toàn này, chứng minh dữ liệu hoàn toàn hợp lý thực tế và không có lỗi nhập liệu 

In [ ]:
# Kiểm tra latitude / longitude
coordinate_rules = {
    "Origin Location Latitude": (-90, 90),
    "Origin Location Longitude": (-180, 180),

    "Destination Location Latitude": (-90, 90),
    "Destination Location Longitude": (-180, 180),

    "Current Location Latitude": (-90, 90),
    "Curren Location Longitude": (-180, 180)}

coordinate_audit = []
for col, (lower, upper) in coordinate_rules.items():
    series = df_raw[col]
    coordinate_audit.append({
        "column": col,
        "missing": series.isna().sum(),
        "zero": series.eq(0).sum(),
        "below_valid_range":
            (series < lower).sum(),
        "above_valid_range":
            (series > upper).sum(),
        "min": series.min(),
        "max": series.max()
    })

coordinate_audit = pd.DataFrame(coordinate_audit)

coordinate_audit

,column,missing,zero,below_valid_range,above_valid_range,min,max
0,Origin Location Latitude,0,0,0,0,9.973636,30.000345
1,Origin Location Longitude,0,0,0,0,72.056000,91.843582
2,Destination Location Latitude,0,0,0,0,8.172701,32.684722
3,Destination Location Longitude,0,0,0,0,70.740636,94.961065
4,Current Location Latitude,1,0,0,0,8.700890,32.367928
5,Curren Location Longitude,1,0,0,0,69.657698,95.529550


>Các trường latitude và longitude đều nằm trong phạm vi tọa độ hợp lệ.  
>Current Location Latitude/Longitude có một giá trị thiếu nhưng không có tọa độ vượt phạm vi hợp lệ.

In [ ]:
# Kiểm tra Minimum Kms To Be Covered In A Day
df_raw["Minimum Kms To Be Covered In A Day"].value_counts(dropna=False)

Minimum Kms To Be Covered In A Day
NaN      2645
250.0     920
275.0      20
Name: count, dtype: int64

>`Minimum Kms To Be Covered In A Day` chủ yếu nhận hai giá trị 250 và 275 km/ngày, đồng thời có tỷ lệ missing rất cao; biến này sẽ được đánh giá lại dựa trên giá trị phân tích trước khi đưa vào analytical dataset.

#### __4.6 Kiểm tra dữ liệu phân loại (categorical)__

In [43]:
CATEGORICAL_COLS = [
    "Gps Provider",
    "Shipment Type",
    "Ontime",
    "Vehicle Type",
    "Customer Name",
    "Supplier Name",
    "Origin Location",
    "Destination Location",
    "Vehicle Registration",
    "Driver Name",
    "Material Shipped",
    "Current Location"
]

categorical_audit = []

for col in CATEGORICAL_COLS:
    raw = df_raw[col]
    normalized = (raw
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.casefold())

    categorical_audit.append({
        "column": col,
        "missing_count": raw.isna().sum(),
        "raw_unique": raw.nunique(dropna=True),
        "normalized_unique": normalized.nunique(dropna=True),
        "possible_duplicate_categories": raw.nunique(dropna=True) - normalized.nunique(dropna=True)})

categorical_audit = pd.DataFrame(categorical_audit)

categorical_audit

,column,missing_count,raw_unique,normalized_unique,possible_duplicate_categories
0,Gps Provider,0,29,29,0
1,Shipment Type,0,2,2,0
2,Ontime,0,2,2,0
3,Vehicle Type,764,38,38,0
4,Customer Name,0,36,36,0
5,Supplier Name,0,193,192,1
6,Origin Location,0,152,124,28
7,Destination Location,0,353,353,0
8,Vehicle Registration,0,1262,1259,3
9,Driver Name,317,1239,1234,5


In [45]:
# Tìm cụ thể category bị tách
def find_category_variants(series):
    temp = pd.DataFrame({"raw_value": series.dropna().astype(str)})
    temp["normalized_value"] = (temp["raw_value"].str.strip().str.replace(r"\s+", " ", regex=True).str.casefold())

    variants = (
        temp
        .groupby("normalized_value")["raw_value"]
        .agg(lambda x: sorted(set(x))))
    return variants[variants.apply(len) > 1]

find_category_variants(
    df_raw["Origin Location"]
)

normalized_value
anekal, bangalore, karnataka                     [Anekal, Bangalore, Karnataka, Anekal, bangalo...
bokaro steel city, bokaro, jharkhand             [Bokaro Steel City, Bokaro, Jharkhand, Bokaro ...
chikatmati, sundergarh, odisha                   [Chikatmati, Sundergarh, Odisha, Chikatmati, s...
degaon tambe, satara, maharashtra                [Degaon Tambe, Satara, Maharashtra, Degaon tam...
durgapur rs, bardhaman, west bengal              [Durgapur Rs, Bardhaman, West Bengal, Durgapur...
embalam, pondicherry, pondicherry                [Embalam, Pondicherry, Pondicherry, Embalam, p...
gopanapalli, krishnagiri, tamil nadu             [Gopanapalli, Krishnagiri, Tamil Nadu, Gopanap...
irungattukottai, kanchipuram, tamil nadu         [Irungattukottai, Kanchipuram, Tamil Nadu, Iru...
jagadambigainagar, tiruvallur, tamil nadu        [Jagadambigainagar, Tiruvallur, Tamil Nadu, Ja...
jamalpur, gurgaon, haryana                       [Jamalpur, Gurgaon, Haryana, Jamalpur, gurg

> Các biến `Shipment Type` và `Ontime` có tập giá trị nhỏ và nhất quán, `Vehical Type`, `Driver Name` bị missing nhiều. Ở cột `Origin Location` raw_unique từ **152** giá trị sau khi chuẩn hóa thì thành **124** giá trị sau khi chuẩn hóa về khoảng trắng và chữ hoa/thường, cho thấy location cần được chuẩn hóa trước Route Analysis.

In [ ]:
# Kiểm tra Supplier Name
find_category_variants(
    df_raw["Supplier Name"]
)

normalized_value
pratiksha freight carriers    [Pratiksha  Freight Carriers, Pratiksha Freigh...
Name: raw_value, dtype: object

> Ở cột `Supplier Name` này thì dữ liệu khác nhau do khoảng trắng vì thế cần chuẩn hóa về một kiểu

In [47]:
# Kiểm tra Vehicle Registration
find_category_variants(
    df_raw["Vehicle Registration"]
)

normalized_value
tn02ap2662    [TN02AP2662, tn02ap2662]
tn18m5881       [TN18M5881, TN18m5881]
tn19ah6070    [TN19AH6070, tn19ah6070]
Name: raw_value, dtype: object

>`Vehicle Registration` có một số phương tiện bị biểu diễn khác nhau do chữ hoa/thường.

In [48]:
df_raw["Shipment Type"].value_counts(dropna=False)

Shipment Type
Regular    3527
Market       58
Name: count, dtype: int64

In [ ]:
df_raw["Ontime"].value_counts(dropna=False)

Ontime
No     2204
Yes    1381
Name: count, dtype: int64

> - `Shipment Type` nhìn chung thì đây là categorical khá sạch
> - `Ontime` chỉ có Yes/No và không có missing. Nhìn bề ngoài thì thấy dữ liệu rất sạch nhưng chưa thể vội kết luận

#####  Kiểm tra Ontime có khớp với ETA hay không?

In [ ]:
ontime_check = df_raw[[
        "Booking ID",
        "Ontime",
        "Booking Date",
        "Planned ETA",
        "Actual ETA",
        "Trip Start Date"]].copy()

datetime_cols = [
    "Booking Date",
    "Planned ETA",
    "Actual ETA",
    "Trip Start Date"]

for col in datetime_cols:
    ontime_check[col] = pd.to_datetime(ontime_check[col],errors="coerce")

In [52]:
# Tính Ontime từ ETA
has_eta = (
    ontime_check['Planned ETA'].notna() &
    ontime_check['Actual ETA'].notna()
)

ontime_check['derived_ontime'] = pd.NA
ontime_check.loc[has_eta, 'derived_ontime'] = np.where(
                                                       ontime_check.loc[has_eta, 'Actual ETA']     
                                                       <=
                                                       ontime_check.loc[has_eta, 'Planned ETA'],
                                                       'Yes','No')

pd.crosstab(
    ontime_check.loc[has_eta, "Ontime"],
    ontime_check.loc[has_eta, "derived_ontime"]
)

derived_ontime,No,Yes
Ontime,,
No,2187,1
Yes,0,1372


In [54]:
mismatch_mask = (has_eta &
    (ontime_check["Ontime"] != ontime_check["derived_ontime"]))

ontime_check.loc[mismatch_mask]

,Booking ID,Ontime,Booking Date,Planned ETA,Actual ETA,Trip Start Date,derived_ontime
3584,WDSBKTP44502,No,2019-04-15 15:15:13,1899-12-30 04:06:00,1899-12-30 03:21:00,1899-12-30,Yes


>Trong 3.560 dòng có đủ Planned ETA và Actual ETA, chỉ có một mismatch giữa raw `Ontime` và trạng thái được suy ra từ ETA; mismatch này thuộc một record có timestamp bất thường năm 1899.

##### Chỉ so sánh trên datetime hợp lệ

In [55]:
valid_timeline = (has_eta
    & ontime_check["Booking Date"].notna()
    & ontime_check["Trip Start Date"].notna()
    & ontime_check["Booking Date"].dt.year.ge(2000)
    & ontime_check["Planned ETA"].dt.year.ge(2000)
    & ontime_check["Actual ETA"].dt.year.ge(2000)
    & ontime_check["Trip Start Date"].dt.year.ge(2000)
    & (ontime_check["Actual ETA"] >= ontime_check["Booking Date"])
    & (ontime_check["Actual ETA"] >= ontime_check["Trip Start Date"]))

In [56]:
pd.crosstab(
    ontime_check.loc[
        valid_timeline,
        "Ontime"
    ],
    ontime_check.loc[
        valid_timeline,
        "derived_ontime"
    ]
)

derived_ontime,No,Yes
Ontime,,
No,2187,0
Yes,0,1364


> Kết luận: Trên những record có timeline hợp lệ, Ontime khớp **100** với logic Planned ETA vs Actual ETA 

In [ ]:
print("Số dòng timeline hợp lệ:",valid_timeline.sum())
print("Số Booking ID duy nhất:",ontime_check.loc[valid_timeline,"Booking ID"].nunique())

Số dòng timeline hợp lệ: 3551
Số Booking ID duy nhất: 3548


In [58]:
df_raw.loc[df_raw["Actual ETA"].isna(),"Ontime"].value_counts()

Ontime
No     16
Yes     9
Name: count, dtype: int64

> 25 dòng thiếu Actual ETA vẫn có raw `Ontime`, nhưng không thể xác minh độc lập bằng ETA. Vì vậy, các KPI delivery performance sẽ sử dụng trạng thái được suy ra từ Planned ETA và Actual ETA đã được validate làm source of truth. Trường raw `Ontime` được sử dụng cho mục đích kiểm tra và reconciliation.

__Kết luận__  
Các vấn đề quan trọng nhất bao gồm:

- Raw dataset chưa hoàn toàn ở grain một dòng cho mỗi booking.
- Một số booking thiếu `Actual ETA`, khiến trạng thái giao hàng không thể được xác định bằng ETA.
- Một số timestamp không hợp lệ về mặt thời gian, bao gồm các giá trị năm 1899 và các trường hợp Actual ETA xảy ra trước Booking Date hoặc Trip Start Date.
- Một số categorical fields chưa được chuẩn hóa, có thể làm cùng một business entity bị chia thành nhiều nhóm khi aggregate.
- Một số dimension như Vehicle Type và Transportation Distance có missing values nhưng các booking tương ứng vẫn có thể sử dụng cho những KPI không phụ thuộc các dimension đó.
- Raw `Ontime` nhìn chung nhất quán với ETA trên các timeline hợp lệ, nhưng ETA đã được validate sẽ được sử dụng làm source of truth cho KPI.
- Không có bằng chứng cho thấy các chuyến có khoảng cách lớn là data error, và các booking có distance bằng 0 có cùng tọa độ Origin/Destination nên chưa bị xem là bất thường.


### __5. Làm sạch dữ liệu__

In [59]:
df = df_raw.copy()

print("Raw shape:", df_raw.shape)
print("Working shape:", df.shape)

Raw shape: (3585, 28)
Working shape: (3585, 28)


#### __5.1 Chuẩn hóa tên cột__

In [63]:
# Danh sách tên cột gốc
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

01. gps_provider
02. booking_id
03. shipment_type
04. booking_date
05. vehicle_registration
06. origin_location
07. destination_location
08. origin_location_latitude
09. origin_location_longitude
10. destination_location_latitude
11. destination_location_longitude
12. data_ping_time
13. planned_eta
14. current_location
15. actual_eta
16. current_location_latitude
17. current_location_longitude
18. ontime
19. trip_start_date
20. trip_end_date
21. transportation_distance_km
22. vehicle_type
23. minimum_kms_to_be_covered_in_a_day
24. driver_name
25. driver_mobile_no
26. customer_name
27. supplier_name
28. material_shipped


In [68]:
def to_snake_case(column_name):   
    column_name = column_name.strip()
    column_name = re.sub(r"[()]", "", column_name)
    column_name = re.sub(r"[^A-Za-z0-9]+", "_",column_name)
    column_name = re.sub(r"_+", "_", column_name)

    return (column_name.strip("_").lower())

raw_columns = df.columns.tolist()

df.columns = [
    to_snake_case(col)
    for col in df.columns
]

In [71]:
# Chỉnh sửa tên cột đặc biệt
COLUMN_NAME_FIXES = {"ontime": "raw_ontime"}
df = df.rename(columns=COLUMN_NAME_FIXES)

clean_columns = df.columns.tolist()

In [ ]:
# Kết quả sau khi chuẩn hóa 
column_name_mapping = pd.DataFrame({
    "raw_column": df_raw.columns,
    "clean_column": df.columns
})

column_name_mapping

,raw_column,clean_column
0,Gps Provider,gps_provider
1,Booking ID,booking_id
2,Shipment Type,shipment_type
3,Booking Date,booking_date
4,Vehicle Registration,vehicle_registration
5,Origin Location,origin_location
6,Destination Location,destination_location
7,Origin Location Latitude,origin_location_latitude
8,Origin Location Longitude,origin_location_longitude
9,Destination Location Latitude,destination_location_latitude


#### __5.2 Chuẩn hóa Missing Values__
Trong phần 4 Kiểm tra dữ liệu, một số giá trị thiếu không được lưu dưới dạng `NaN` mà được biểu diễn bằng chuỗi text như `"NULL"`.  

Điều này khiến các hàm như `isna()` không phát hiện đầy đủ missing values.
Mục tiêu của bước này:

- Chuẩn hóa các missing-like values về `NaN`.
- Không thực hiện imputation.
- Không xóa booking.
- Kiểm tra số lượng missing trước và sau khi chuẩn hóa.

In [73]:
# Kiểm tra lại missing dạng text trên dataframe đã rename
TEXT_COLS = df.select_dtypes(include=["object", "string"]).columns
NULL_LIKE_VALUES = ["NULL"]

null_like_before = {}
for col in TEXT_COLS:
    normalized = (df[col].astype("string").str.strip().str.upper())
    count = normalized.isin(NULL_LIKE_VALUES).sum()

    if count > 0:
        null_like_before[col] = count
pd.Series(null_like_before,name="null_like_count").sort_values(ascending=False)

current_location    12
gps_provider         1
Name: null_like_count, dtype: int64

In [74]:
# Chuẩn hóa khoảng trắng 
for col in TEXT_COLS:
    df[col] = (df[col].astype("string").str.strip())

In [76]:
# Chuyển NULL thành missing value
for col in TEXT_COLS:
    null_mask = (df[col].str.upper().eq("NULL"))
    df.loc[null_mask,col] = pd.NA

In [77]:
# Check lại sau khi cleaning
null_like_after = {}

for col in TEXT_COLS:
    normalized = (df[col].astype("string").str.strip().str.upper())
    count = normalized.eq("NULL").sum()
    if count > 0:
        null_like_after[col] = count

pd.Series(null_like_after,dtype="int64",name="null_like_count")

Series([], Name: null_like_count, dtype: int64)

> Không còn `NULL` trong các text fields

In [78]:
# So sánh missing trước và sau
missing_before = (df_raw.isna().sum())
missing_after = (df.isna().sum())

missing_comparison = pd.DataFrame({
    "missing_before": missing_before.values,
    "missing_after": missing_after.values
})

missing_comparison.index = df.columns
missing_comparison[missing_comparison["missing_after"] != missing_comparison["missing_before"]]

,missing_before,missing_after
gps_provider,0,1
current_location,0,12


> Trước đó đã có `NAN` thật thì missing_after sẽ bằng NAN ban đầu + 12 NULL text, tương tự cho gps_provider

__Kết luận__
- Các missing values được biểu diễn dưới dạng chuỗi `"NULL"` đã được chuẩn hóa thành missing value thực sự.
- 12 giá trị `"NULL"` trong `current_location` đã được chuyển thành missing.
- 1 giá trị `"NULL"` trong `gps_provider` đã được chuyển thành missing.
- Các text fields được loại bỏ khoảng trắng thừa ở đầu và cuối.

#### __5.2 Chuẩn hóa dữ liệu phân loại__
Bước kiểm tra dữ liệu cho thấy một số biến phân loại có nhiều cách biểu diễn cho cùng một business entity do:
- Khác chữ hoa và chữ thường.
- Có nhiều khoảng trắng liên tiếp.
- Có khoảng trắng thừa.
- Vehicle Registration không thống nhất chữ hoa/thường.

Mục tiêu của bước này là chuẩn hóa cách biểu diễn categorical values nhưng vẫn giữ nguyên ý nghĩa nghiệp vụ.

In [79]:
# Kiểm tra unique trước cleaning
CATEGORICAL_CLEAN_COLS = [
    "gps_provider",
    "shipment_type",
    "raw_ontime",
    "vehicle_registration",
    "origin_location",
    "destination_location",
    "vehicle_type",
    "customer_name",
    "supplier_name",
    "driver_name",
    "material_shipped",
    "current_location"
]

unique_before = (
    df[CATEGORICAL_CLEAN_COLS]
    .nunique(dropna=True)
    .rename("unique_before")
)

unique_before

gps_provider              28
shipment_type              2
raw_ontime                 2
vehicle_registration    1262
origin_location          152
destination_location     353
vehicle_type              38
customer_name             36
supplier_name            193
driver_name             1234
material_shipped         712
current_location        1708
Name: unique_before, dtype: int64

In [80]:
# Chuẩn hóa khoảng trắng
for col in CATEGORICAL_CLEAN_COLS:
    df[col] = (df[col].astype("string").str.strip().str.replace(r"\s+"," ",regex=True))

In [ ]:
# Tạo hàm canonicalize category để chuẩn hóa danh mục
def canonicalize_category(series):
    cleaned = (series.astype("string").str.strip().str.replace(r"\s+"," ",regex=True))
    normalized_key = (cleaned.str.casefold())
    temp = pd.DataFrame({
        "value": cleaned,
        "key": normalized_key})
    canonical_map = (temp.dropna().groupby("key")["value"].agg(
            lambda x: x.value_counts().index[0]))

    return normalized_key.map(canonical_map).astype("string")

In [82]:
CANONICALIZE_COLS = [
    "gps_provider",
    "origin_location",
    "destination_location",
    "vehicle_type",
    "customer_name",
    "supplier_name",
    "driver_name",
    "material_shipped",
    "current_location"
]

for col in CANONICALIZE_COLS:
    df[col] = canonicalize_category(df[col])

In [83]:
# Chuẩn hóa vehicle_registration
df["vehicle_registration"] = (df["vehicle_registration"].str.upper())

In [85]:
# Chuẩn hóa raw_ontime
ONTIME_MAP = {
    "yes": "Yes",
    "no": "No"}

df["raw_ontime"] = (df["raw_ontime"].str.casefold().map(ONTIME_MAP).astype("string"))

df["raw_ontime"].value_counts(dropna=False)

raw_ontime
No     2204
Yes    1381
Name: count, dtype: Int64

In [87]:
# Chuẩn hóa shipment_type
SHIPMENT_TYPE_MAP = {
    "regular": "Regular",
    "market": "Market"}

df["shipment_type"] = (df["shipment_type"].str.casefold().map(SHIPMENT_TYPE_MAP).astype("string"))

df["shipment_type"].value_counts(dropna=False)

shipment_type
Regular    3527
Market       58
Name: count, dtype: Int64

In [88]:
# So sánh unique trước và sau
unique_after = (df[CATEGORICAL_CLEAN_COLS]
    .nunique(dropna=True)
    .rename("unique_after"))

categorical_cleaning_audit = pd.concat(
    [unique_before,unique_after],axis=1)

categorical_cleaning_audit["categories_merged"] =(
    categorical_cleaning_audit["unique_before"]-categorical_cleaning_audit["unique_after"])

categorical_cleaning_audit

,unique_before,unique_after,categories_merged
gps_provider,28,28,0
shipment_type,2,2,0
raw_ontime,2,2,0
vehicle_registration,1262,1259,3
origin_location,152,124,28
destination_location,353,353,0
vehicle_type,38,38,0
customer_name,36,36,0
supplier_name,193,192,1
driver_name,1234,1234,0


> `origin_location` thay đổi khá mạnh vì trước đó nhiều địa điểm chỉ khác chữ hoa/thường.

In [89]:
# Kiểm tra location
df["origin_location"].nunique()

124

> Sau khi chuẩn hóa thì còn **124** trong khi dữ liệu raw còn **152**, điều đó không có nghĩa là xóa **28 location** mà là số lượng unique Origin Location giảm sau chuẩn hóa do nhiều location trước đó chỉ khác cách viết hoa/thường.

In [90]:
# Validate missing không bị fill
missing_after_category_cleaning = (df[CATEGORICAL_CLEAN_COLS].isna().sum())
missing_after_category_cleaning

gps_provider              1
shipment_type             0
raw_ontime                0
vehicle_registration      0
origin_location           0
destination_location      0
vehicle_type            764
customer_name             0
supplier_name             0
driver_name             317
material_shipped          0
current_location         12
dtype: int64

In [91]:
print(
    "Vehicle Type missing:",
    df["vehicle_type"].isna().sum()
)

print(
    "GPS Provider missing:",
    df["gps_provider"].isna().sum()
)

Vehicle Type missing: 764
GPS Provider missing: 1


In [92]:
print("df_raw:",df_raw.shape)
print("df:",df.shape)

df_raw: (3585, 28)
df: (3585, 28)


> __Tóm lại:__ Việc giảm số category không làm mất record mà chỉ hợp nhất các cách biểu diễn khác nhau của cùng một business entity.

In [ ]:

# Chuẩn hóa chuỗi "NULL"/"NA"/"-" -> NaN thật, trim khoảng trắng
for c in TEXT_COLS:
    df[c] = df[c].astype("string").str.strip()
    df[c] = df[c].mask(df[c].str.match(NULL_LIKE, na=False))
for c in DATE_COLS:
    df[c] = pd.to_datetime(df[c], errors="coerce")
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["ontime_flag"] = df["ontime_flag"].str.title()

# distance_km = 0 là vô lý -> coi như thiếu
df.loc[df["distance_km"] == 0, "distance_km"] = np.nan

# Bo 2 cột không dùng
df = df.drop(columns=["min_kms_per_day", "driver_mobile"], errors="ignore")

# Gán "Unknown" cho biến phân loại để nhóm
for c in ["vehicle_type", "driver_name"]:
    df[c] = df[c].fillna("Unknown")

miss = df.isna().sum(); miss = miss[miss > 0].sort_values(ascending=False)
print("Dữ liệu còn thiếu sau khi làm sạch:"); print(miss.to_string() if len(miss) else "(Không còn)")

>Nhận xét chất lượng dữ liệu:

Bộ dữ liệu khá sạch về mặt cấu trúc — các cột khóa (`booking_id`, `origin`, `destination`, `ontime_flag`, `customer_name`, `supplier_name`) đều đầy đủ 100%. Vấn đề thiếu dữ liệu tập trung ở vài cột và được xử lý theo 3 hướng:

- Bỏ cả cột (thiếu quá nhiều / không giá trị phân tích): min_kms_per_day thiếu ~74% và driver_mobile là PII → loại bỏ.
- Giữ NaN, loại khỏi phép tính liên quan (biến đo lường — tuyệt đối không điền để tránh bịa số): actual_eta thiếu 25 dòng → 25 chuyến không tính được delay; distance_km thiếu 148 dòng và có giá trị 0 bất hợp lý → quy về NaN, loại khi tính tốc độ.
Gán "Unknown" (biến phân loại để nhóm): vehicle_type (764) và driver_name (317) → giữ lại các chuyến này trong biểu đồ thay vì để chúng biến mất.
- Về trùng lặp: nếu chỉ xét trùng trên mọi cột thì df.duplicated().sum() = 0. Nhưng xét theo khóa nghiệp vụ booking_id thì có 2 mã trùng / 3 dòng thừa — thực chất là các chuyến chở nhiều loại hàng cho nhiều khách (consolidated), không phải bản sao lỗi. Vì grain đã chốt là 1 dòng = 1 chuyến, ta gộp về 1 dòng/booking (ảnh hưởng < 0,1% dữ liệu).

Kết luận: dữ liệu đủ tin cậy để phân tích. 

### __5. Feature Engineering - Tạo các cột để phân tích__

Dữ liệu thô chỉ ghi lại sự kiện (các mốc thời gian, tọa độ, tên đối tác), chưa trả lời được câu hỏi phân tích. Ví dụ: file gốc có `planned_eta` và `actual_eta`, nhưng **không có cột nào nói "chuyến này trễ bao lâu"** — muốn phân tích delay thì buộc phải tự tính. Bước 3 biến dữ liệu thô thành các cột **đo lường được** và **nhóm được**, chia làm 3 loại:

- **Metric (đo lường)** — không có sẵn, phải tính: `delay_hours` (= `actual_eta − planned_eta`), `trip_duration_hours`, `avg_speed_kmph`.
- **Flag / phân loại** — để đếm tỷ lệ & gán nhãn: `is_delayed` (0/1, dùng tính *delay rate*), `delivery_status`, `delay_level` (Slight/Medium/Severe), `is_delay_outlier`.
- **Dimension (chiều cắt lát)** — để nhóm dữ liệu: `route` (ghép origin + destination), `booking_month`, `booking

In [ ]:
WEEKDAY = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

df['delay_hours'] = ((df['actual_eta'] - df['planned_eta']).dt.total_seconds() / 3600).round(2)
df['is_delayed'] = np.where(df['delay_hours'] > 0, 1, 0)
df.loc[df['delay_hours'].isna(), 'is_delayed'] = np.nan 
df['delivery_status'] = np.select(
    [df['delay_hours'].isna(), df['delay_hours'] > 0],
    ['Unknown', 'Delayed'], default='On Time')

def lvl(h):
    if pd.isna(h): return 'Unknown'
    if h <= 0: return 'On Time'
    if h <= 24: return 'Slight Delay'
    if h <= 72: return 'Medium Delay'
    return 'Severe Delay'

# Mức độ delay
df['delay_level'] = df['delay_hours'].apply(lvl)
lo, hi = df['delay_hours'].quantile([0.01, 0.99])
df['is_delayed_outlier'] = ((df['delay_hours'] < lo )| (df['delay_hours'] > hi)).astype('Int64')

# Tuyến đường
df['route'] = df['origin'].fillna('Unknown') + '->' + df['destination'].fillna('Unknown')

# Thời gian chuyến đi
df['trip_duration_hours'] = ((df['trip_end'] - df['trip_start']).dt.total_seconds() / 3600).round(2)
ok = (df['trip_duration_hours'] > 0) & (df['distance_km'] > 0)

# Tốc độ trung bình
df['avg_speed_kmph'] = np.where(ok, (df['distance_km'] / df['trip_duration_hours']).round(2), np.nan)

# Thời gian booking theo tháng và tuần
df["booking_month"]   = df["booking_date"].dt.to_period("M").dt.to_timestamp()
df["booking_weekday"] = pd.Categorical(df["booking_date"].dt.day_name(), categories=WEEKDAY, ordered=True)

print("Số cột sau feature engineering:", df.shape[1])
df[["booking_id","route","distance_km","delay_hours","delivery_status","delay_level"]].head()



Số cột sau feature engineering: 36


,booking_id,route,distance_km,delay_hours,delivery_status,delay_level
0,AEIBK2027469,"Shive, Pune, Maharashtra->Pondur, Kanchipuram,...",1290.0,-51.26,On Time,On Time
1,VCV00014153/082021,"Daimler India Commercial Vehicles,Kanchipuram,...",29.0,-79.13,On Time,On Time
2,VCV00014063/082021,"Daimler India Commercial Vehicles,Kanchipuram,...",21.0,-78.82,On Time,On Time
3,VCV00014741/082021,"Daimler India Commercial Vehicles,Kanchipuram,...",20.0,-89.01,On Time,On Time
4,AEIBK2027446,"Khorajnanoda, Ahmedabad, Gujarat->Singaperumal...",1900.0,-30.76,On Time,On Time


### __6. Kiểm chứng dữ liệu__

In [ ]:
ct = pd.crosstab(df['ontime_flag'], df['delivery_status'])
ct

delivery_status,Delayed,On Time,Unknown
ontime_flag,,,
No,2187,1,16
Yes,0,1372,9


In [ ]:
match = ((df.ontime_flag=="Yes")&(df.delivery_status=="On Time")).sum() + \
        ((df.ontime_flag=="No")&(df.delivery_status=="Delayed")).sum()
valid = df.delay_hours.notna().sum()
print(f"\nKhớp {match}/{valid} dòng có ETA hợp lệ ({match/valid*100:.1f}%)")


Khớp 3559/3560 dòng có ETA hợp lệ (100.0%)


> Nhận xét: khớp ~99,9%. → Có thể dùng delay_hours làm chuẩn.